# Part 1: Dogs vs Cats Under Tile Permutations
This notebook trains three image-classification architecture families on Dogs vs Cats.
It evaluates how fixed tile-wise permutations affect validation accuracy.
Results are aggregated by tile count and plotted as accuracy vs number of tiles.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [ ]:
import sys, os
from pathlib import Path


### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


prep for local imports

In [ ]:
def get_project_root() -> str:
    """Resolve the project root in local notebooks, Drive-backed Colab, or cloned Colab runs."""
    try:
        from google.colab import drive  # type: ignore

        drive.mount('/content/drive')
        drive_root = Path('/content/drive/MyDrive/MLDS_Final_Project')
        if drive_root.exists():
            return str(drive_root)
    except ImportError:
        pass
    except Exception as exc:
        print(f'Google Drive was not mounted automatically: {exc}')

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').exists():
            return str(candidate)
    return str(current)

def install_project_requirements_for_colab(project_root: Path) -> None:
    """Install non-PyTorch dependencies in Colab without replacing CUDA-matched torch wheels."""
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return

    import subprocess

    requirements_path = project_root / 'requirements.txt'
    filtered_requirements = Path('/tmp/mlds_colab_requirements.txt')
    skip_prefixes = ('torch', 'torchvision')
    filtered_lines = [
        line
        for line in requirements_path.read_text().splitlines()
        if not line.strip().lower().startswith(skip_prefixes)
    ]
    filtered_requirements.write_text('\n'.join(filtered_lines) + '\n')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(filtered_requirements)])

# Add project root to sys.path so we can import modules from src
REPO_PATH = get_project_root()
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

os.chdir(REPO_PATH)
install_project_requirements_for_colab(Path(REPO_PATH))
print(f"Project root: {REPO_PATH}")


make local imports

In [ ]:
import importlib
from src.utils.config import CVExperimentConfig
import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
import src.training.experiment_steps as experiment_steps
experiment_steps = importlib.reload(experiment_steps)
experiment_output_paths = experiment_results.experiment_output_paths
get_device = experiment_results.get_device
load_experiment_samples = experiment_results.load_experiment_samples
plot_accuracy_vs_tiles = experiment_results.plot_accuracy_vs_tiles
save_aggregated_accuracy = experiment_results.save_aggregated_accuracy
save_rows = experiment_results.save_rows
collect_model_permutation_results = experiment_steps.collect_model_permutation_results
from src.preprocessing.dogs_cats import class_counts
from src.preprocessing.permutations import build_permutation_records
from src.utils.io import ensure_dir, save_csv
from src.utils.plotting import plot_permutation_samples
from src.utils.reproducibility import seed_everything

### Setup configs

In [ ]:
configs = CVExperimentConfig()
display(configs)

### make installations before final external imports

In [ ]:
# Dependencies are installed during the setup/bootstrap cell above when running in Colab.
print('Dependency setup is complete.')


### final imports (after doing pip install if working on colab)

In [ ]:
import json
import random
from IPython.display import Image, display
import pandas as pd
import numpy as np
import torch


## Experiments

### Experiment helpers
Shared helpers are kept in `src/evaluation/experiment_results.py`; notebook-specific helpers stay here.

In [ ]:
def get_part1_output_paths(*, config: CVExperimentConfig) -> dict[str, str]:
    """Build stable Part 1 output paths for notebook display."""
    paths = experiment_output_paths(
        results_dir=config.results_dir,
        figures_dir=config.figures_dir,
        part_name='part1',
    )
    paths['accuracy_plot'] = paths['figure']
    return paths

### Setup Exp

In [ ]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [ ]:
output_paths = get_part1_output_paths(config=configs)
output_paths

### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
train_samples, validation_samples, test_samples = load_experiment_samples(config=configs, seed=configs.seed)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print(f'Test samples: {len(test_samples)}')
print('Train class counts:', class_counts(samples=train_samples))
print('Validation class counts:', class_counts(samples=validation_samples))
print('Test class counts:', class_counts(samples=test_samples))

### Experiments - Run Baselines
Run the configured baseline grid/model/permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [ ]:
# Build and save permutation records
permutation_records = build_permutation_records(
    grid_sizes=configs.grid_sizes,
    num_permutations=configs.num_permutations,
    seed=configs.seed,
    include_identity=True,
)
permutation_rows = [record.__dict__ | {'permutation': json.dumps(record.permutation)} for record in permutation_records]
save_csv(data=permutation_rows, path=output_paths['permutations'])
print(f"Saved {len(permutation_records)} permutation records")

if configs.plot_samples:
    import matplotlib.pyplot as plt

    plot_permutation_samples(
        samples=train_samples,
        permutation_records=permutation_records,
        image_size=configs.image_size,
    )
    plt.show()


In [ ]:
# Setup reproducibility and load data
seed = configs.seed
seed_everything(seed=seed, deterministic=configs.deterministic)
# Data already loaded above
print(f"Loaded {len(train_samples)} train, {len(validation_samples)} val samples")

In [ ]:
# Prepare shared result accumulation across model runs
all_rows = []
run_id = configs.config_name

### Train ResNet-50
ResNet-50 is a convolutional residual network from the ResNet family. It uses skip connections to train a deeper CNN reliably and serves as a strong convolutional baseline.

In [ ]:
model_name = "resnet50"
device = get_device(config=configs)
rows = collect_model_permutation_results(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training ResNet-50 with {len(rows)} runs")

### Train DeiT-Small
DeiT-Small comes from the data-efficient vision transformer family. It represents images as patch tokens and uses self-attention in a compact ViT-style architecture.

In [ ]:
model_name = "deit_small"
rows = collect_model_permutation_results(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training DeiT-Small with {len(rows)} runs")

### Train MLP-Mixer
MLP-Mixer is an all-MLP vision architecture. It alternates token-mixing and channel-mixing MLP layers, using neither convolution nor self-attention.

In [ ]:
model_name = "mlp_mixer"
rows = collect_model_permutation_results(
    config=configs,
    model_name=model_name,
    run_id=run_id,
    train_samples=train_samples,
    validation_samples=validation_samples,
    permutation_records=permutation_records,
    seed=seed,
    device=device,
)
all_rows.extend(rows)
save_rows(rows=all_rows, output_path=output_paths['raw_results'])
print(f"Completed training MLP-Mixer with {len(rows)} runs")

In [ ]:
# Aggregate results and plot
raw_results = pd.read_csv(filepath_or_buffer=output_paths['raw_results'])
aggregated_results = save_aggregated_accuracy(
    raw_results=raw_results,
    group_columns=['model_name', 'grid_size', 'num_tiles'],
    output_path=output_paths['aggregated_results'],
)
plot_accuracy_vs_tiles(aggregated=aggregated_results, output_path=output_paths['accuracy_plot'])
aggregated_results

### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk. Each row averages all permutation scores for the same model and tile count.


In [ ]:
saved_results = {
    'raw': pd.read_csv(filepath_or_buffer=output_paths['raw_results']),
    'aggregated': pd.read_csv(filepath_or_buffer=output_paths['aggregated_results']),
}
display(saved_results['aggregated'])

### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
figure_path = output_paths['accuracy_plot']
if Path(figure_path).exists():
    display(Image(filename=figure_path))
else:
    print(f'Plot not found yet: {figure_path}')
